In [1]:
!pip install pandas openpyxl xlsxwriter 
!pip install google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2
!pip install time logging

## Set up and install all the required packages
import pandas as pd
import time, logging
import os
import io
import re
from concurrent.futures import ThreadPoolExecutor

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from googleapiclient.http import MediaIoBaseDownload

print(os.getcwd())
print(os.listdir())
    
# --- Step 1: Authenticate with OAuth ---
SCOPES = ['https://www.googleapis.com/auth/drive']
    
creds = None
if os.path.exists('token.json'):
    creds = Credentials.from_authorized_user_file('token.json', SCOPES)
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file('/mnt/data/credentials.json', SCOPES)
        #creds = flow.run_console() --> Testing Purpose
        creds = flow.run_local_server()
        #creds = flow.run_local_server(port=0, open_browser=False) --> Testing Purpose
        
        with open('token.json', 'w') as token:
            token.write(creds.to_json())

#drive_service = build('drive', 'v3', credentials=creds)

if not os.path.exists('token.json'):
    raise FileNotFoundError("Missing token.json! Run your Google login cell first.")

creds = Credentials.from_authorized_user_file('token.json', SCOPES)
drive_service = build('drive', 'v3', credentials=creds)

# --- Step 2a: Load and Clean Input ---
#df = pd.read_excel("Product-mix-Report_Final.xlsx")

# PASTE YOUR GOOGLE DRIVE FILE ID HERE 
FILE_ID = '1hZSF-UaF9M01vQ1u6R09EWadqpurHKaS'

# Stream the file directly from Google into Python's memory
request = drive_service.files().get_media(fileId=FILE_ID)
file_stream = io.BytesIO()
downloader = MediaIoBaseDownload(file_stream, request)

done = False
while done is False:
    status, done = downloader.next_chunk()
    
# Rewind the stream and hand it off to Pandas
file_stream.seek(0)
df = pd.read_excel(file_stream, engine='openpyxl')

df = df.rename(columns={
    "Item Number": "sku",
    "Item supplier (Partner)": "partner",
    "Sales outlets": "location",
    "Sales volume": "sales",
    "Current Inventory": "inventory",
    "Revenue": "revenue"
})

df['partner'] = df['partner'].astype(str).str.strip().str.lower()

# --- Step 2b: Partner Name Normalistation ---
partner_map = {
    "happiness initiative": "Happiness Initiative",
    "riau candle": "Riau Candle",
    "palette puzzles": "Palette Puzzles",
    "the everyday club": "The Everyday Club",
    # add more mappings here
}
df['partner'] = df['partner'].map(partner_map).fillna(df['partner'])

# --- Step 3: Event Classification ---
def classify_event(row):
    if row['sales'] > 0:
        return "Sales"
    elif "GRN" in str(row['location']):
        return "GRN"
    elif "Adjust" in str(row['location']):
        return "Inventory Adjustment"
    elif "Transfer" in str(row['location']):
        return "Location Transfer"
    elif "Opening" in str(row['location']):
        return "Opening Balance"
    elif "Closing" in str(row['location']):
        return "Closing Balance"
    else:
        return "Ambiguous"

df['event_class'] = df.apply(classify_event, axis=1)

# --- Step 4: Generate Partner Reports ---
folder_id = "1SFjlr1Lxy_fWB407QEda8bbPwnKH1QpB"   # replace with your Drive folder ID
logging.basicConfig(filename="upload_log.txt", level=logging.INFO)


def safe_filename(name: str) -> str:
    #Always cast to string first
    name = str(name)
    # Replace unsafe characters with underscores
    cleaned = re.sub(r"[^\w\s-]", "_", name)
    # Replace spaces with underscores, lowercase for consistency
    return cleaned.strip().replace(" ", "_").lower() + "_report.xlsx"

def upload_file(filename, folder_id, retries=3):
    for attempt in range(retries):
        try:
            query = f"name='{filename}' and '{folder_id}' in parents and trashed=false"
            results = drive_service.files().list(q=query, fields="files(id, name)").execute()
            items = results.get('files', [])
            file_metadata = {'name': filename, 'parents': [folder_id]}
            media = MediaFileUpload(filename, mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

            if items:
                file_id = items[0]['id']
                uploaded = drive_service.files().update(
                    fileId=file_id, body=file_metadata, media_body=media, fields='id'
                ).execute()
                logging.info(f"Overwritten {filename} with ID {uploaded.get('id')}")
            else:
                uploaded = drive_service.files().create(
                    body=file_metadata, media_body=media, fields='id'
                ).execute()
                logging.info(f"Uploaded new {filename} with ID {uploaded.get('id')}")
            return uploaded
        except Exception as e:
            logging.error(f"Attempt {attempt+1} failed for {filename}: {e}")
            time.sleep(2)
    logging.error(f"Failed to upload {filename} after {retries} attempts")
    return None

def generate_partner_report(p):
    partner_data = df[df['partner'] == p]
    filename = safe_filename(p)
    
    with pd.ExcelWriter(filename, engine="xlsxwriter") as writer:
        for loc in ["Kreta Ayer", "Potong Pasir", "Online"]:
            loc_data = partner_data[partner_data['location'].str.contains(loc, case=False, na=False)]
            if not loc_data.empty:
                loc_report = loc_data.groupby(
                    ["sku", "location", "event_class"]
                ).agg({
                    "sales": "sum",
                    "inventory": "sum",
                    "revenue": "sum"
                }).reset_index()

                # Add opening/closing balances
                opening = loc_data.groupby("sku")["inventory"].first().reset_index()
                opening["event_class"] = "Opening Balance"
                closing = loc_data.groupby("sku")["inventory"].last().reset_index()
                closing["event_class"] = "Closing Balance"
                loc_report = pd.concat([loc_report, opening, closing], ignore_index=True)

                loc_report.to_excel(writer, sheet_name=loc, index=False)

        # Summary sheet
        summary = partner_data.groupby("event_class").agg({
            "sales": "sum",
            "inventory": "sum",
            "revenue": "sum"
        }).reset_index()
        summary.to_excel(writer, sheet_name="Summary", index=False)

        # Flags sheet for ambiguous entries
        flags = partner_data[partner_data['event_class'] == "Ambiguous"]
        if not flags.empty:
            flags.to_excel(writer, sheet_name="Flags", index=False)

    upload_file(filename, folder_id)

# --- Step 5: Run in Parallel for Performance --- (ThreadPoolExecutor can run multiple at a time)
partners = df['partner'].unique()
## This Code Requires High RAM --> Have Faster Performace
#with ThreadPoolExecutor(max_workers=2) as executor:
    #executor.map(generate_partner_report, partners)

## This code will run smoothly --> Have slower Performance
for p in partners:
    generate_partner_report(p)

# --- Step 5b: Check what missing excel files are there
results = drive_service.files().list(
    q=f"'{folder_id}' in parents and trashed=false",
    fields="files(name)"
).execute()

drive_files = [f['name'] for f in results.get('files', [])]
local_files = [f for f in os.listdir() if f.endswith('.xlsx')]

missing = [f for f in local_files if f not in drive_files]
print("Missing from Drive:", missing)

# --- Completed the AI workflow ---
print("")
print("End of AI workflow")
print("")

'''# --- Step 6: Validation Against Sample Output ---
sample = pd.read_excel("Riau_Candles_stock_movement_only.xlsx", sheet_name=None)
for sheet, df_sample in sample.items():
    print(f"Sample sheet {sheet} columns:", df_sample.columns.tolist())
'''


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement time (from versions: none)
ERROR: No matching distribution found for time

[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
/home/jovyan
['.profile', '.bash_logout', '.bashrc', 'green_kulture_report.xlsx', 'doodle_dat_report.xlsx', 'neuhabitat_report.xlsx', 'little_botany_report.xlsx', 'made51_report.xlsx', 'jomo_studio_report.xlsx', 'will & well_report.xlsx', 'soapnut republic_report.xlsx', 'Palette Puzzles_report.xlsx', '.ipython', 'all_things_felt_report.xlsx', 'helping_hands_penan_report.xlsx', 'alwis & xavier_report.xlsx', 'alika_day_report.xlsx', 'brambe_report.xlsx', 'innerfyre_report.xlsx', 'helping hands penan_report.xlsx', 'poasia

'# --- Step 6: Validation Against Sample Output ---\nsample = pd.read_excel("Riau_Candles_stock_movement_only.xlsx", sheet_name=None)\nfor sheet, df_sample in sample.items():\n    print(f"Sample sheet {sheet} columns:", df_sample.columns.tolist())\n'